In [2]:
import pandas as pd
import numpy as np
import requests

In [3]:
url1 = "https://en.wikipedia.org/wiki/List_of_countries_by_greenhouse_gas_emissions"


headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

response = requests.get(url1, headers=headers)

tables = pd.read_html(response.text)
print(f"Found {len(tables)} tables")

for i in tables:
    print(i)


/tmp/ipykernel_545/2045813852.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


Found 18 tables
    0                                                  1
0 NaN  This article needs to be updated. The reason g...
    Country/territory/region/group    Emissions                            \
    Country/territory/region/group         1970         1990         2000   
0                            Aruba        36.71       219.98       347.85   
1                      Afghanistan     15437.43     12561.60     13915.67   
2                           Angola     18998.76     31940.04     67023.25   
3                         Anguilla         3.36         7.90        17.88   
4                          Albania      8224.85     11493.15      7179.74   
..                             ...          ...          ...          ...   
207                       Zimbabwe     21723.58     34348.03     32470.94   
208                   GLOBAL TOTAL  24002751.74  32726228.02  36175146.34   
209                 European Union   4591422.70   4877284.15   4481448.73   
210                  Ai

In [4]:
url2 = "https://en.wikipedia.org/wiki/List_of_countries_by_carbon_dioxide_emissions"


headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

response = requests.get(url2, headers=headers)

tables = pd.read_html(response.text)
print(f"Found {len(tables)} tables")


for j in tables:
    print(j)




/tmp/ipykernel_545/3291743775.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


Found 18 tables
    0                                                  1
0 NaN  This article needs to be updated. The reason g...
                                         Location % of global total  \
                                         Location % of global total   
0                                           World              100%   
1                                           China             34.0%   
2                                   United States             12.0%   
3                                           India              7.6%   
4                      European Union post-Brexit              6.5%   
..                                            ...               ...   
207                     Saint Pierre and Miquelon           0.0001%   
208                                      Anguilla           0.0001%   
209                              Falkland Islands           0.0000%   
210  Saint Helena, Ascension and Tristan da Cunha           0.0000%   
211               

In [5]:

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}


# CO2 page
url1 = "https://en.wikipedia.org/wiki/List_of_countries_by_carbon_dioxide_emissions"
response1 = requests.get(url1, headers=headers)
tables_co2 = pd.read_html(response1.text)
print(f"CO2 page: found {len(tables_co2)} tables")

# GHG page
url2 = "https://en.wikipedia.org/wiki/List_of_countries_by_greenhouse_gas_emissions"
response2 = requests.get(url2, headers=headers)
tables_ghg = pd.read_html(response2.text)
print(f"GHG page: found {len(tables_ghg)} tables")

/tmp/ipykernel_545/965895971.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables_co2 = pd.read_html(response1.text)


CO2 page: found 18 tables
GHG page: found 18 tables


/tmp/ipykernel_545/965895971.py:15: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables_ghg = pd.read_html(response2.text)


In [6]:
import pandas as pd

# ---- CO2 page tables ----
# tables_co2[1]  -> Fossil emissions by location, 2023 vs 2000, % of global total
# tables_co2[2]  -> Fossil CO2 emissions 2023 + per-capita (the money table)
# tables_co2[3]  -> Cumulative fossil CO2 emissions 1970-2023

co2_2023 = tables_co2[2].copy()
co2_2023.columns = ['co2_2023_kt', 'country', 'co2_per_capita_2023']
co2_2023 = co2_2023[co2_2023['country'] != 'Global Total']  # drop the world row, keep separately if you want it

co2_cumulative = tables_co2[3].copy()
co2_cumulative.columns = ['co2_cumulative_1970_2023_kt', 'country', 'co2_2023_kt_dup']
co2_cumulative = co2_cumulative[['country', 'co2_cumulative_1970_2023_kt']]

# ---- GHG page tables ----
# tables_ghg[1]  -> full historical GHG table, 1970-2023 + % global + % change + per capita (richest one)
# tables_ghg[2]  -> GHG emissions 2023 + per-capita
# tables_ghg[3]  -> Cumulative GHG emissions 1970-2023

ghg_hist = tables_ghg[1].copy()
ghg_hist.columns = ['country','y1970','y1990','y2000','y2010','y2020','y2021','y2022','y2023',
                     'pct_global_2023','pct_change_1990_2023','ghg_per_capita_2023']

ghg_hist['pct_change_1990_2023'] = (
    ghg_hist['pct_change_1990_2023'].astype(str).str.replace('−','-').str.replace('%','').astype(float)
)
ghg_hist['pct_global_2023'] = ghg_hist['pct_global_2023'].astype(str).str.replace('%','').astype(float)

# ---- Merge CO2 + GHG on country ----
dashboard_df = (
    ghg_hist[['country','y2023','pct_global_2023','pct_change_1990_2023','ghg_per_capita_2023']]
    .rename(columns={'y2023':'ghg_2023_kt'})
    .merge(co2_2023, on='country', how='outer')
    .merge(co2_cumulative, on='country', how='outer')
)

dashboard_df = dashboard_df[~dashboard_df['country'].isin(['Global Total','European Union'])]  # keep these separate as aggregates
dashboard_df.to_csv('climate_dashboard_data.csv', index=False)
dashboard_df.sort_values('ghg_2023_kt', ascending=False).head(20)

,country,ghg_2023_kt,pct_global_2023,pct_change_1990_2023,ghg_per_capita_2023,co2_2023_kt,co2_per_capita_2023,co2_cumulative_1970_2023_kt
73,GLOBAL TOTAL,52962900.86,100.000,61.84,6.59,NaN,NaN,NaN
40,China,15943986.55,30.104,311.31,11.11,13259638.95,9.24,2.779245e+08
206,United States,5960804.38,11.255,-4.00,17.61,4682039.41,13.83,2.761683e+08
93,India,4133554.36,7.805,198.87,2.90,2955181.68,2.07,5.948510e+07
159,Russia,2672039.44,5.045,-12.84,18.66,2069502.01,14.45,9.865735e+07
26,Brazil,1300168.87,2.455,93.60,5.97,479504.23,2.20,1.685767e+07
94,Indonesia,1200199.79,2.266,202.24,4.29,674535.94,2.41,1.526786e+07
104,Japan,1041012.82,1.966,-20.97,8.31,944758.61,7.54,6.067946e+07
97,Iran,996752.68,1.882,199.58,11.64,778802.30,9.10,1.933123e+07
168,Saudi Arabia,805158.11,1.520,240.39,22.17,622913.60,17.15,1.620520e+07


In [14]:
dashboard_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 214 entries, 0 to 215
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   country                      214 non-null    object 
 1   ghg_2023_kt                  211 non-null    float64
 2   pct_global_2023              211 non-null    float64
 3   pct_change_1990_2023         211 non-null    float64
 4   ghg_per_capita_2023          211 non-null    object 
 5   co2_2023_kt                  208 non-null    float64
 6   co2_per_capita_2023          208 non-null    float64
 7   co2_cumulative_1970_2023_kt  210 non-null    float64
dtypes: float64(6), object(2)
memory usage: 15.0+ KB


In [7]:
# Normalize country name casing/whitespace on both tables BEFORE merging
for df in [ghg_hist, co2_2023, co2_cumulative]:
    df['country'] = df['country'].astype(str).str.strip()
    df['country'] = df['country'].replace({'GLOBAL TOTAL': 'Global Total'})

non_country_rows = [
    'Global Total', 'European Union', 'International Aviation',
    'International Shipping', 'Sea transport', 'Air transport'
]

ghg_hist_clean = ghg_hist[~ghg_hist['country'].isin(non_country_rows)].copy()
ghg_hist_clean = ghg_hist_clean.rename(columns={'y2023': 'ghg_2023_kt'})   # <-- the missing rename

co2_2023_clean = co2_2023[~co2_2023['country'].isin(non_country_rows)].copy()
co2_cumulative_clean = co2_cumulative[~co2_cumulative['country'].isin(non_country_rows)].copy()

dashboard_df = (
    ghg_hist_clean[['country','ghg_2023_kt','pct_global_2023','pct_change_1990_2023','ghg_per_capita_2023']]
    .merge(co2_2023_clean, on='country', how='outer')
    .merge(co2_cumulative_clean, on='country', how='outer')
)

global_co2_2023 = dashboard_df['co2_2023_kt'].sum()
dashboard_df['co2_pct_global_2023'] = (dashboard_df['co2_2023_kt'] / global_co2_2023 * 100).round(3)

print(f"Remaining rows with any NaN: {dashboard_df.isna().any(axis=1).sum()} / {len(dashboard_df)}")
print(dashboard_df.isna().sum())

dashboard_df.to_csv('climate_dashboard_data.csv', index=False)
dashboard_df.sort_values('ghg_2023_kt', ascending=False).head(15)

Remaining rows with any NaN: 2 / 209
country                        0
ghg_2023_kt                    1
pct_global_2023                1
pct_change_1990_2023           1
ghg_per_capita_2023            1
co2_2023_kt                    1
co2_per_capita_2023            1
co2_cumulative_1970_2023_kt    1
co2_pct_global_2023            1
dtype: int64


,country,ghg_2023_kt,pct_global_2023,pct_change_1990_2023,ghg_per_capita_2023,co2_2023_kt,co2_per_capita_2023,co2_cumulative_1970_2023_kt,co2_pct_global_2023
39,China,15943986.55,30.104,311.31,11.11,13259638.95,9.24,2.779245e+08,35.054
199,United States,5960804.38,11.255,-4.00,17.61,4682039.41,13.83,2.761683e+08,12.378
89,India,4133554.36,7.805,198.87,2.90,2955181.68,2.07,5.948510e+07,7.813
153,Russia,2672039.44,5.045,-12.84,18.66,2069502.01,14.45,9.865735e+07,5.471
25,Brazil,1300168.87,2.455,93.60,5.97,479504.23,2.20,1.685767e+07,1.268
90,Indonesia,1200199.79,2.266,202.24,4.29,674535.94,2.41,1.526786e+07,1.783
98,Japan,1041012.82,1.966,-20.97,8.31,944758.61,7.54,6.067946e+07,2.498
91,Iran,996752.68,1.882,199.58,11.64,778802.30,9.10,1.933123e+07,2.059
162,Saudi Arabia,805158.11,1.520,240.39,22.17,622913.60,17.15,1.620520e+07,1.647
33,Canada,747678.03,1.412,28.49,19.39,575011.96,14.91,2.683242e+07,1.520


In [16]:
dashboard_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209 entries, 0 to 208
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   country                      209 non-null    object 
 1   ghg_2023_kt                  208 non-null    float64
 2   pct_global_2023              208 non-null    float64
 3   pct_change_1990_2023         208 non-null    float64
 4   ghg_per_capita_2023          208 non-null    object 
 5   co2_2023_kt                  208 non-null    float64
 6   co2_per_capita_2023          208 non-null    float64
 7   co2_cumulative_1970_2023_kt  208 non-null    float64
 8   co2_pct_global_2023          208 non-null    float64
dtypes: float64(7), object(2)
memory usage: 14.8+ KB


In [8]:
dashboard_df.isnull().sum()

,0
country,0
ghg_2023_kt,1
pct_global_2023,1
pct_change_1990_2023,1
ghg_per_capita_2023,1
co2_2023_kt,1
co2_per_capita_2023,1
co2_cumulative_1970_2023_kt,1
co2_pct_global_2023,1


In [9]:
dashboard_df["ghg_per_capita_2023"] = dashboard_df['ghg_per_capita_2023'].astype(float)

In [10]:
dashboard_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209 entries, 0 to 208
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   country                      209 non-null    object 
 1   ghg_2023_kt                  208 non-null    float64
 2   pct_global_2023              208 non-null    float64
 3   pct_change_1990_2023         208 non-null    float64
 4   ghg_per_capita_2023          208 non-null    float64
 5   co2_2023_kt                  208 non-null    float64
 6   co2_per_capita_2023          208 non-null    float64
 7   co2_cumulative_1970_2023_kt  208 non-null    float64
 8   co2_pct_global_2023          208 non-null    float64
dtypes: float64(8), object(1)
memory usage: 14.8+ KB


In [11]:
dashboard_df = dashboard_df.dropna()

In [12]:
dashboard_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 207 entries, 0 to 208
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   country                      207 non-null    object 
 1   ghg_2023_kt                  207 non-null    float64
 2   pct_global_2023              207 non-null    float64
 3   pct_change_1990_2023         207 non-null    float64
 4   ghg_per_capita_2023          207 non-null    float64
 5   co2_2023_kt                  207 non-null    float64
 6   co2_per_capita_2023          207 non-null    float64
 7   co2_cumulative_1970_2023_kt  207 non-null    float64
 8   co2_pct_global_2023          207 non-null    float64
dtypes: float64(8), object(1)
memory usage: 16.2+ KB


In [13]:
dashboard_df.describe()

,ghg_2023_kt,pct_global_2023,pct_change_1990_2023,ghg_per_capita_2023,co2_2023_kt,co2_per_capita_2023,co2_cumulative_1970_2023_kt,co2_pct_global_2023
count,2.070000e+02,207.000000,207.000000,207.000000,2.070000e+02,207.000000,2.070000e+02,207.000000
mean,2.498346e+05,0.471715,132.056329,6.929082,1.827308e+05,4.721304,6.666826e+06,0.483082
std,1.232532e+06,2.327173,321.120401,8.820291,1.008423e+06,6.810964,2.883772e+07,2.665947
min,2.127000e+01,0.000000,-77.690000,0.540000,2.100000e+00,0.040000,8.833000e+01,0.000000
25%,5.365565e+03,0.010500,4.765000,2.050000,2.322340e+03,0.915000,6.393383e+04,0.006000
50%,3.315078e+04,0.063000,84.820000,4.520000,1.174405e+04,2.740000,4.280268e+05,0.031000
75%,1.023233e+05,0.193500,172.040000,7.970000,6.003506e+04,5.800000,2.846443e+06,0.158500
max,1.594399e+07,30.104000,3674.970000,65.290000,1.325964e+07,62.590000,2.779245e+08,35.054000


In [18]:
from google.colab import files
dashboard_df.to_csv('climate_dashboard_data.csv', index=False)
files.download('climate_dashboard_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
ghg_hist.columns.tolist()

['country',
 'y1970',
 'y1990',
 'y2000',
 'y2010',
 'y2020',
 'y2021',
 'y2022',
 'y2023',
 'pct_global_2023',
 'pct_change_1990_2023',
 'ghg_per_capita_2023']

In [15]:
year_cols = ['y1970','y1990','y2000','y2010','y2020','y2021','y2022','ghg_2023_kt']
year_labels = [1970, 1990, 2000, 2010, 2020, 2021, 2022, 2023]

trend_df = ghg_hist_clean.melt(
    id_vars='country',
    value_vars=year_cols,
    var_name='year_col',
    value_name='ghg_kt'
)
trend_df['Year'] = trend_df['year_col'].map(dict(zip(year_cols, year_labels)))
trend_df = trend_df.drop(columns='year_col')

major = ['China','United States','India','Russia','Japan']
trend_df_top = trend_df[trend_df['country'].isin(major)]

trend_df_top.to_csv('emissions_trend.csv', index=False)
trend_df_top.head(10)

,country,ghg_kt,Year
35,China,2035096.63,1970
89,India,792811.36,1970
98,Japan,1016283.47,1970
159,Russia,1723159.42,1970
196,United States,5788649.39,1970
243,China,3876387.77,1990
297,India,1383063.84,1990
306,Japan,1317217.96,1990
367,Russia,3065696.36,1990
404,United States,6209294.71,1990


In [16]:
print(ghg_hist_clean.columns.tolist())
print(ghg_hist_clean.shape)

year_cols = ['y1970','y1990','y2000','y2010','y2020','y2021','y2022','ghg_2023_kt']
year_labels = [1970, 1990, 2000, 2010, 2020, 2021, 2022, 2023]

trend_df = ghg_hist_clean.melt(
    id_vars='country',
    value_vars=year_cols,
    var_name='year_col',
    value_name='ghg_kt'
)

['country', 'y1970', 'y1990', 'y2000', 'y2010', 'y2020', 'y2021', 'y2022', 'ghg_2023_kt', 'pct_global_2023', 'pct_change_1990_2023', 'ghg_per_capita_2023']
(208, 12)


In [19]:
files.download('emissions_trend.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>